# Protein Function Prediction — CNN Model
**Project:** COMP 3608 B-rank mission  
**This notebook:** Loads processed data from `data/processed/` → builds a 1D CNN → tunes hyperparameters → evaluates with full metrics, confusion matrices, and heatmaps.

In [1]:
import warnings, random, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from collections import Counter

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from tensorflow.keras.utils import to_categorical

# Keras Tuner
import keras_tuner as kt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROC_DIR = Path('data/processed')
FIG_DIR  = Path('figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

ModuleNotFoundError: No module named 'tensorflow'

## 1. Load Processed Data

In [ ]:
df = pd.read_csv(PROC_DIR / 'processed_sequences.csv')
class_names = np.load(PROC_DIR / 'class_names.npy', allow_pickle=True)
MAX_LEN  = int(np.load(PROC_DIR / 'max_len.npy'))

print(f'Dataset shape : {df.shape}')
print(f'Max seq length: {MAX_LEN}')
print(f'Classes ({len(class_names)}): {class_names}')
df['label'].value_counts()

### 1.1 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vc = df['label'].value_counts()
colors = sns.color_palette('husl', len(vc))

# Bar chart
axes[0].bar(vc.index, vc.values, color=colors, edgecolor='black', linewidth=0.6)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Protein Function Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=9)

# Pie chart
axes[1].pie(vc.values, labels=vc.index, autopct='%1.1f%%', colors=colors, startangle=140, pctdistance=0.8)
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.2 Sequence Length Distribution

In [ ]:
df['seq_len'] = df['sequence'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['seq_len'], bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].axvline(MAX_LEN, color='red', linestyle='--', label=f'95th pct = {MAX_LEN}')
axes[0].set_title('Overall Sequence Length Distribution')
axes[0].set_xlabel('Length')
axes[0].set_ylabel('Count')
axes[0].legend()

for label in df['label'].unique():
    axes[1].hist(df[df['label'] == label]['seq_len'], bins=40,
                 alpha=0.5, label=label)
axes[1].set_title('Sequence Length by Class')
axes[1].set_xlabel('Length')
axes[1].set_ylabel('Count')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig(FIG_DIR / 'sequence_lengths.png', dpi=150, bbox_inches='tight')
plt.show()
print(df['seq_len'].describe())

## 2. Sequence Encoding

In [ ]:
# Integer encoding  (0 = pad, 1-20 = amino acids, 21 = unknown 'X')
AA_ORDER = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i + 1 for i, aa in enumerate(AA_ORDER)}  # 1-indexed
AA_TO_IDX['X'] = 21   # unknown / non-standard
VOCAB_SIZE = 22       # 0 (pad) + 20 AA + 1 unknown

def encode_sequence(seq, max_len):
    """Truncate-or-pad and integer-encode a sequence."""
    seq = seq[:max_len]
    enc = [AA_TO_IDX.get(aa, 21) for aa in seq.upper()]
    pad = [0] * (max_len - len(enc))
    return enc + pad   # post-padding

print(f'Encoding {len(df)} sequences to length {MAX_LEN}...')
X_raw = np.array([encode_sequence(s, MAX_LEN) for s in df['sequence']], dtype=np.int32)

# Encode labels
le = LabelEncoder()
le.classes_ = class_names
y_int  = le.transform(df['label'])
NUM_CLASSES = len(class_names)
y_cat  = to_categorical(y_int, NUM_CLASSES)

print(f'X shape: {X_raw.shape}')
print(f'y shape: {y_cat.shape}')
print(f'Classes : {class_names}')

## 3. Train / Validation / Test Split

In [ ]:
X_temp, X_test, y_temp, y_test, yi_temp, yi_test = train_test_split(
    X_raw, y_cat, y_int,
    test_size=0.15, stratify=y_int, random_state=SEED
)
X_train, X_val, y_train, y_val, yi_train, yi_val = train_test_split(
    X_temp, y_temp, yi_temp,
    test_size=0.15 / 0.85, stratify=yi_temp, random_state=SEED
)

print(f'Train : {X_train.shape[0]}')
print(f'Val   : {X_val.shape[0]}')
print(f'Test  : {X_test.shape[0]}')

# Class weights (handle imbalance)
cw = compute_class_weight('balanced', classes=np.unique(yi_train), y=yi_train)
class_weight_dict = dict(enumerate(cw))
print('Class weights:', {class_names[k]: round(v, 3) for k, v in class_weight_dict.items()})

## 4. CNN Architecture

In [ ]:
def build_cnn(hp=None,
              embed_dim=64,
              num_conv_blocks=3,
              filters_start=64,
              kernel_sizes=(3, 5, 7),
              dense_units=128,
              dropout_rate=0.4,
              l2_reg=1e-4,
              learning_rate=1e-3):
    """
    1-D CNN for protein function prediction.
    Accepts either plain values OR a keras_tuner HyperParameters object.
    """
    if hp is not None:
        embed_dim       = hp.Int('embed_dim',       min_value=32,   max_value=128, step=32)
        num_conv_blocks = hp.Int('num_conv_blocks',  min_value=2,    max_value=4)
        filters_start   = hp.Choice('filters_start', [32, 64, 128])
        kernel_sizes    = (
            hp.Choice('kernel_1', [3, 5]),
            hp.Choice('kernel_2', [5, 7]),
            hp.Choice('kernel_3', [7, 11]),
        )
        dense_units     = hp.Choice('dense_units',  [64, 128, 256])
        dropout_rate    = hp.Float('dropout_rate',   min_value=0.2, max_value=0.6, step=0.1)
        l2_reg          = hp.Choice('l2_reg',        [1e-5, 1e-4, 1e-3])
        learning_rate   = hp.Choice('learning_rate', [1e-4, 5e-4, 1e-3])

    reg = regularizers.l2(l2_reg)

    # ── Input & Embedding ──────────────────────────────────────────────────
    inp = keras.Input(shape=(MAX_LEN,), name='sequence_input')
    x = layers.Embedding(input_dim=VOCAB_SIZE,
                            output_dim=embed_dim,
                            mask_zero=True,
                            name='aa_embedding')(inp)

    # ── Multi-scale Parallel Conv Block (first) ────────────────────────────
    branches = []
    for k in kernel_sizes:
        b = layers.Conv1D(filters_start, kernel_size=k, padding='same',
                          activation='relu', kernel_regularizer=reg)(x)
        b = layers.BatchNormalization()(b)
        branches.append(b)
    x = layers.Concatenate()(branches)
    x = layers.Dropout(dropout_rate)(x)

    # ── Stacked Conv Blocks ────────────────────────────────────────────────
    filters = filters_start * len(kernel_sizes)   # after concat
    for i in range(1, num_conv_blocks):
        filters_next = min(filters * 2, 512)
        residual = layers.Conv1D(filters_next, kernel_size=1, padding='same',
                                 kernel_regularizer=reg)(x)   # projection shortcut

        x = layers.Conv1D(filters_next, kernel_size=3, padding='same',
                          activation='relu', kernel_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv1D(filters_next, kernel_size=3, padding='same',
                          kernel_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Add()([x, residual])   # residual connection
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling1D(pool_size=2)(x)
        x = layers.Dropout(dropout_rate)(x)
        filters = filters_next

    # ── Global Pooling + Classifier ────────────────────────────────────────
    avg = layers.GlobalAveragePooling1D()(x)
    mx  = layers.GlobalMaxPooling1D()(x)
    x   = layers.Concatenate()([avg, mx])

    x = layers.Dense(dense_units, activation='relu', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(dense_units // 2, activation='relu', kernel_regularizer=reg)(x)
    x = layers.Dropout(dropout_rate / 2)(x)

    out = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)

    model = keras.Model(inputs=inp, outputs=out, name='ProteinCNN')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                 keras.metrics.AUC(multi_label=False, name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

# Quick sanity check
model_check = build_cnn()
model_check.summary()

### 4.1 Architecture Visualisation

In [ ]:
# Layer-wise parameter heatmap
layer_data = []
for layer in model_check.layers:
    params = layer.count_params()
    if params > 0:
        layer_data.append({'Layer': layer.name, 'Type': type(layer).__name__,
                           'Parameters': params})

ldf = pd.DataFrame(layer_data)
fig, ax = plt.subplots(figsize=(12, max(4, len(ldf) * 0.35)))
pivot = ldf.set_index('Layer')[['Parameters']]
sns.heatmap(pivot, annot=True, fmt=',d', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Parameter count'}, ax=ax)
ax.set_title('CNN Parameter Distribution by Layer', fontsize=14, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(FIG_DIR / 'model_param_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hyperparameter Tuning

In [ ]:
# Reduce dataset size for tuning if very large (speeds up search)
MAX_TUNE_SAMPLES = 5000
if len(X_train) > MAX_TUNE_SAMPLES:
    idx = np.random.choice(len(X_train), MAX_TUNE_SAMPLES, replace=False)
    X_tune, y_tune = X_train[idx], y_train[idx]
    yi_tune = yi_train[idx]
else:
    X_tune, y_tune, yi_tune = X_train, y_train, yi_train

print(f'Tuning on {len(X_tune)} samples')

tuner = kt.Hyperband(
    build_cnn,
    objective='val_accuracy',
    max_epochs=20,
    factor=3,
    directory='tuner_results',
    project_name='protein_cnn',
    overwrite=True,
    seed=SEED
)

stop_early = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

tuner.search(
    X_tune, y_tune,
    epochs=20,
    validation_data=(X_val, y_val),
    class_weight=class_weight_dict,
    callbacks=[stop_early],
    verbose=1
)

best_hp = tuner.get_best_hyperparameters(1)[0]
print('\n=== Best Hyperparameters ===')
for k, v in best_hp.values.items():
    print(f'  {k:20s}: {v}')

## 6. Train Best Model

In [ ]:
model = tuner.hypermodel.build(best_hp)

cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10,
                             restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=4, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint('best_protein_cnn.keras',
                              monitor='val_accuracy', save_best_only=True, verbose=1)
]

EPOCHS = 60
BATCH  = 64

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    class_weight=class_weight_dict,
    callbacks=cb_list,
    verbose=1
)
print('Training complete.')

## 7. Training Curves

In [ ]:
def plot_history(history):
    hist = history.history
    metrics = ['loss', 'accuracy', 'auc', 'precision', 'recall']
    available = [m for m in metrics if m in hist]
    n = len(available)

    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1: axes = [axes]

    for ax, metric in zip(axes, available):
        train_vals = hist[metric]
        val_vals   = hist.get(f'val_{metric}', [])
        epochs_range = range(1, len(train_vals) + 1)

        ax.plot(epochs_range, train_vals, 'b-o', ms=3, label='Train')
        if val_vals:
            ax.plot(epochs_range, val_vals, 'r-o', ms=3, label='Val')
        ax.set_title(metric.capitalize(), fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle('Training History', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history)

## 8. Evaluation on Test Set

In [ ]:
model = keras.models.load_model('best_protein_cnn.keras')

y_pred_prob = model.predict(X_test, batch_size=128, verbose=0)
y_pred_int  = np.argmax(y_pred_prob, axis=1)

test_metrics = model.evaluate(X_test, y_test, batch_size=128, verbose=0)
metric_names = model.metrics_names
print('\n=== Test Set Metrics ===')
for name, val in zip(metric_names, test_metrics):
    print(f'  {name:15s}: {val:.4f}')

print('\n=== Classification Report ===')
print(classification_report(yi_test, y_pred_int, target_names=class_names, digits=4))

### 8.1 Metrics Summary Table

In [ ]:
report_dict = classification_report(
    yi_test, y_pred_int,
    target_names=class_names,
    output_dict=True
)
report_df = pd.DataFrame(report_dict).T
report_df = report_df.drop(['accuracy'], errors='ignore')
report_df[['precision', 'recall', 'f1-score']] = report_df[['precision', 'recall', 'f1-score']].astype(float)

fig, ax = plt.subplots(figsize=(10, max(4, len(report_df) * 0.5)))
sns.heatmap(
    report_df[['precision', 'recall', 'f1-score']],
    annot=True, fmt='.3f', cmap='Blues',
    linewidths=0.5, vmin=0, vmax=1, ax=ax,
    cbar_kws={'label': 'Score'}
)
ax.set_title('Per-Class Metrics Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'metrics_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(yi_test, y_pred_int)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2f'],
    ['Raw Confusion Matrix', 'Normalised Confusion Matrix']
):
    sns.heatmap(
        data, annot=True, fmt=fmt,
        xticklabels=class_names, yticklabels=class_names,
        cmap='Blues' if 'Raw' in title else 'YlGnBu',
        linewidths=0.5, ax=ax
    )
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True', fontsize=11)
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.3 ROC Curves (One-vs-Rest)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
colors = sns.color_palette('tab10', NUM_CLASSES)

for i, (cname, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve((yi_test == i).astype(int), y_pred_prob[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{cname}  (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — One-vs-Rest', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

macro_auc = roc_auc_score(y_test, y_pred_prob, average='macro', multi_class='ovr')
print(f'Macro-average ROC-AUC: {macro_auc:.4f}')

### 8.4 Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for i, (cname, color) in enumerate(zip(class_names, colors)):
    prec, rec, _ = precision_recall_curve((yi_test == i).astype(int), y_pred_prob[:, i])
    ap = average_precision_score((yi_test == i).astype(int), y_pred_prob[:, i])
    ax.plot(rec, prec, lw=2, color=color, label=f'{cname}  (AP = {ap:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(FIG_DIR / 'pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.5 Prediction Confidence Heatmap

In [ ]:
# Average predicted probability per true class → shows how confidently
# the model assigns each class and where it "leaks" confidence
n_show = min(200, len(yi_test))
idx = np.random.choice(len(yi_test), n_show, replace=False)
sort_order = np.argsort(yi_test[idx])

prob_matrix = y_pred_prob[idx][sort_order]   # shape (n_show, NUM_CLASSES)
true_labels = yi_test[idx][sort_order]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(prob_matrix.T, aspect='auto', cmap='hot', vmin=0, vmax=1,
               interpolation='nearest')
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(class_names)
ax.set_xlabel('Sample index (sorted by true class)', fontsize=11)
ax.set_ylabel('Predicted Class', fontsize=11)
ax.set_title('Prediction Confidence Heatmap', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Probability')

# Draw class boundary lines
boundaries = np.where(np.diff(true_labels))[0] + 0.5
for b in boundaries:
    ax.axvline(b, color='cyan', linewidth=1, alpha=0.7)

plt.tight_layout()
plt.savefig(FIG_DIR / 'confidence_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Feature Importance — Amino Acid Composition vs Predictions

In [ ]:
# Compute AAC for test set
AA_ORDER = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX_FEAT = {aa: i for i, aa in enumerate(AA_ORDER)}

def aac(seq):
    counts = np.zeros(20)
    for aa in seq:
        if aa in AA_TO_IDX_FEAT:
            counts[AA_TO_IDX_FEAT[aa]] += 1
    return counts / len(seq) if len(seq) > 0 else counts

test_seqs = df.iloc[yi_test.tolist() if hasattr(yi_test, 'tolist') else list(yi_test)].reset_index(drop=True)
# Re-extract correctly via the split indices
all_seqs = df['sequence'].values

# Build AAC for test sequences using original df
# We'll use y_pred_int to build a mean AAC per predicted class
mean_aac_per_class = np.zeros((NUM_CLASSES, 20))
test_df_seqs = df['sequence'].values  # all seqs

# Approximate: use a random subset from df per class
for i, cname in enumerate(class_names):
    cls_seqs = df[df['label'] == cname]['sequence'].values[:200]
    if len(cls_seqs) > 0:
        mean_aac_per_class[i] = np.mean([aac(s) for s in cls_seqs], axis=0)

aac_df = pd.DataFrame(mean_aac_per_class, index=class_names, columns=list(AA_ORDER))

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(aac_df, annot=True, fmt='.3f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean Frequency'})
ax.set_title('Mean Amino Acid Composition per Functional Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Amino Acid')
ax.set_ylabel('Protein Class')
plt.tight_layout()
plt.savefig(FIG_DIR / 'aac_class_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()